In [ ]:
# Copy Gemma 3 12B IT tokenizer and chat-template files (no weights) to /kaggle/working,
# and check them against the hashes FT1 job (a) recorded on 2026-11-12.
import hashlib, json, shutil
from pathlib import Path
SRC = Path("/kaggle/input/models/google/gemma-3/transformers/gemma-3-12b-it/1")
OUT = Path("/kaggle/working/gemma-3-12b-it-tokenizer")
EXPECTED = {
    "tokenizer.json": "7d4046bf0505a327dd5a0abbb427ecd4fc82f99c2ceaa170bc61ecde12809b0c",
    "tokenizer.model": "1299c11d7cf632ef3b4e11937501358ada021bbdf7c47638d13c0ee982f2e79c",
    "tokenizer_config.json": "10a27b015a8c561aee8fb959b949e63b6c11b67965c0eccf519e6375439f092e",
    "special_tokens_map.json": "2f7b0adf4fb469770bb1490e3e35df87b1dc578246c5e7e6fc76ecf33213a397",
    "added_tokens.json": "50b2f405ba56a26d4913fd772089992252d7f942123cc0a034d96424221ba946",
}
EXTRA = ["chat_template.json", "chat_template.jinja", "LICENSE", "README.md"]
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
listing = sorted((p.name, p.stat().st_size) for p in SRC.iterdir() if p.is_file())
print("model folder:", listing)
OUT.mkdir(parents=True, exist_ok=False)
report = {"source": str(SRC), "listing": listing, "files": {}, "mismatches": []}
for name in list(EXPECTED) + EXTRA:
    p = SRC / name
    if not p.exists():
        if name in EXPECTED: report["mismatches"].append(name + ": missing")
        continue
    if p.stat().st_size > 50_000_000: raise SystemExit("refusing large file " + name)
    shutil.copy2(p, OUT / name)
    h = sha(OUT / name); report["files"][name] = h
    if name in EXPECTED and h != EXPECTED[name]: report["mismatches"].append(name + ": hash differs")
report["passed"] = not report["mismatches"]
(Path("/kaggle/working") / "export-report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
if not report["passed"]: raise SystemExit("tokenizer export failed")
